# Quick SLM — 04c · SFT corpus top-up

Notebook 04b salvages what has already been generated; it never loads the teacher.
This one is the other half: it **generates additional examples** for the categories
that came out under-represented, and it guarantees the additions are not duplicates
of what is already on disk.

The realised v1 corpus drifted a long way from its design shares:

| category | designed | realised | drift |
|---|---:|---:|---:|
| single_stage | 40% | 57.5% | 1.44x |
| multi_stage | 25% | 31.3% | 1.25x |
| state_memory_conflict | **20%** | **5.7%** | **0.28x** |
| traps | 10% | 2.5% | 0.25x |
| refusals | 5% | 2.9% | 0.59x |

**Read section 3 before generating anything.** Most of that deficit cannot be
generated away: the scenario sources are exhausted, not the teacher budget. This
notebook measures the real headroom first and refuses to plan past it, because
generating past capacity buys duplicates at full teacher price.

## 1 · Framework

Same install as notebook 04. A GPU runtime is needed for section 5 only; sections
1 to 4 are CPU-only and are the ones that decide whether section 5 is worth running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EXTRAS = 'sft'
DRIVE_ROOT = '/content/drive/MyDrive/quick-slm'

import subprocess, sys, importlib
from pathlib import Path

root = Path(DRIVE_ROOT)
candidates = [root / 'code', root, root / 'quick-slm']
REPO_DIR = next((p for p in candidates if (p / 'pyproject.toml').exists()), None)
if REPO_DIR is None:
    raise RuntimeError('No pyproject.toml on Drive under ' + str(root / 'code') + '; upload the repo there and re-run.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR) + '[' + EXTRAS + ']'], check=True)

framework_dir = REPO_DIR / 'framework'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))
importlib.invalidate_caches()

import v1.quick_slm_trainer as q
print('quick-slm-trainer', q.__version__, 'from', Path(q.__file__).parent)
# Refuse to run outside the support window this training version declares in
# training/v1/framework.json. A framework newer than v1's window would not fail
# loudly, it would build a different corpus.
if hasattr(q, 'require_framework'):
    q.require_framework('v1', REPO_DIR)
else:
    raise RuntimeError(
        'quick-slm-trainer ' + q.__version__ + ' predates the support-window check; '
        'training v1 requires >=1.0. See SUPPORT.md.'
    )

## 2 · What survives today

Read the raw shards, apply the corrected validation and the real dedup pass, and
count what actually reaches the packer per category. This is the same funnel
notebook 04b reports, repeated here because the top-up plan is computed from its
output rather than from the design shares.

In [ ]:
import collections
from v1.quick_slm_trainer import Layout, sft_v1
from v1.quick_slm_trainer.sft import load_and_validate
from v1.quick_slm_trainer.sft.corpus import category_histogram
from v1.quick_slm_trainer.sft.dedup import dedup_examples
from v1.quick_slm_trainer.sft.specs import CATEGORIES

layout = Layout().mkdirs_sft()
cfg = sft_v1()

examples, stats, _ = load_and_validate(layout, cfg.sft)
surviving = dedup_examples(examples, threshold=cfg.sft.dedup_jaccard,
                           num_perm=cfg.sft.minhash_perms, progress=True)

have = category_histogram(surviving)
total = sum(have.values())
shares = cfg.sft.category_shares

print()
print('after validation and dedup: ' + f'{total:,}' + ' examples')
print()
print('category'.ljust(24) + 'have'.rjust(8) + 'design'.rjust(9) + 'actual'.rjust(9) + 'drift'.rjust(8))
print('-' * 58)
for k in CATEGORIES:
    n = have.get(k, 0)
    d = shares[k]
    print(k.ljust(24) + f'{n:>8,}' + f'{d:>8.0%}' + f'{n/total:>9.1%}' + f'{n/total/d:>7.2f}x')

## 3 · Headroom: what can actually still be made

The deficit and the *achievable* top-up are different numbers, and confusing them
is how teacher hours get spent on examples dedup will delete.

Two different ceilings apply.

**`state_memory_conflict` is bounded by the spec table.** Its examples are built
from counterfactual swap specs, not from free-text seeds, so the number of pairs
that can ever be told apart is a property of `conflict_specs.py`. `train_capacity()`
counts them exactly, on CPU, with no teacher.

**Everything else is bounded by its seed pool.** Each `(domain, subtype)` cell draws
from a hand-written pool, and past some yield per seed the teacher can only rephrase
a seed it already has. That yield is now measurable, because a full run has happened:
distinct survivors divided by pool size is what the teacher actually delivered.

The table below reports both, and the top-up plan in section 4 is capped by it.

In [ ]:
from v1.quick_slm_trainer.sft.conflict import train_capacity
from v1.quick_slm_trainer.sft.prompts import usable_seeds

PAIRED = 'state_memory_conflict'

def seed_pool(key):
    spec = CATEGORIES[key]
    return sum(len(usable_seeds(key, d, st)) for d in spec.domains for st in spec.subtypes_for(d))

# Measured yield per seed, from the run that just happened.
yield_per_seed = {k: (have.get(k, 0) / seed_pool(k)) for k in CATEGORIES if not CATEGORIES[k].paired}

print('category'.ljust(24) + 'have'.rjust(8) + 'ceiling'.rjust(9) + 'headroom'.rjust(10) + '   bound by')
print('-' * 72)
headroom = {}
for k in CATEGORIES:
    n = have.get(k, 0)
    if k == PAIRED:
        ceiling = train_capacity() * 2          # pairs -> examples, both branches
        why = 'conflict_specs.py spec table'
    else:
        # The teacher has already shown what it yields per seed. Asking for more
        # than it delivered last time buys duplicates, so that yield IS the ceiling.
        ceiling = int(seed_pool(k) * yield_per_seed[k])
        why = f'{seed_pool(k)} seeds x {yield_per_seed[k]:.0f} distinct/seed (measured)'
    room = max(0, ceiling - n)
    headroom[k] = room
    print(k.ljust(24) + f'{n:>8,}' + f'{ceiling:>9,}' + f'{room:>10,}' + '   ' + why)

print()
print('Design shares would need:')
for k in CATEGORIES:
    want = int(have['single_stage'] / shares['single_stage'] * shares[k])
    gap = want - have.get(k, 0)
    verdict = 'reachable' if gap <= headroom[k] else 'NOT REACHABLE from the current sources'
    if gap > 0:
        print('  ' + k.ljust(24) + f'+{gap:>6,} to hit design   ' + verdict)

### What the headroom table means

If a category reads **NOT REACHABLE**, no amount of teacher time closes that gap.
The fix is upstream, in code rather than in generation:

- **`state_memory_conflict`** needs more swap specs in `conflict_specs.py`. Three
  specs currently supply most of the distinct capacity and the smallest supplies
  eight pairs. Adding specs raises the ceiling; regenerating against the same table
  does not.
- **`traps` and `refusals`** need wider seed pools in `prompts.py`. Their measured
  yield is already at what the teacher can distinguish from 36 and 30 seeds.

Before spending anything on the paired category, fix the reject its own audit
predicted: `validate._MARKUP` rejects any `think` containing `<state`, while
`build_conflict_prompt` hands the teacher a prompt full of literal `<state>` tags.
That cost 481 rejects last run, and **each one kills both branches of a pair**.

## 4 · Plan the top-up

Three things keep the additions from colliding with what is already on disk.

1. **A different plan seed.** `plan_requests` is deterministic, so re-running it with
   the seed v1 used would re-draw the same seed topics and tool lists.
2. **Distinct request ids.** `run_generation` resumes by skipping ids already in the
   shard, so a re-used id would be silently skipped rather than generated. The
   top-up namespaces every id with `-t2-`, which also makes the shard self-documenting
   about which pass produced a record.
3. **A different generation seed**, so the teacher samples differently even where a
   seed topic repeats.

None of that guarantees distinct *output*. Section 6 is what guarantees that.

In [ ]:
import dataclasses
from v1.quick_slm_trainer.sft.generate import plan_requests

TOPUP_TAG = 't2'          # bump for a third pass
PLAN_SEED_TOPUP = 20240915
GEN_SEED_TOPUP = 815_493

# Ask only for what section 3 says exists. Zero is a valid answer.
want = {}
for k in CATEGORIES:
    target = int(have['single_stage'] / shares['single_stage'] * shares[k])
    want[k] = max(0, min(target - have.get(k, 0), headroom[k]))

print('planning top-up (capped by measured headroom):')
for k, n in want.items():
    print('  ' + k.ljust(24) + f'{n:>7,}')
print()

topup_cfg = dataclasses.replace(
    cfg.sft,
    target_examples=max(1, sum(want.values())),
    category_shares={k: (v / max(1, sum(want.values()))) for k, v in want.items()},
    # The caps that shaped v1's plan still apply; they are what keeps this from
    # asking for more per seed than the pool can realise.
)

raw = plan_requests(topup_cfg, seed=PLAN_SEED_TOPUP)
raw = [r for r in raw if want.get(r.category, 0) > 0]

# Re-namespace every id (and pair id) so nothing collides with the first pass.
requests = []
for i, r in enumerate(raw):
    new = {'id': f'{r.category}-{TOPUP_TAG}-{i:06d}'}
    if getattr(r, 'is_paired', False) and getattr(r, 'pair_id', None):
        new['pair_id'] = f'{TOPUP_TAG}-{r.pair_id}'
    requests.append(dataclasses.replace(r, **new))

print(f'{len(requests):,} requests planned')
print(collections.Counter(r.category for r in requests))
assert not any(r.id.count(TOPUP_TAG) != 1 for r in requests), 'id namespacing failed'

## 5 · Generate

Appends to the same raw shards notebook 04 wrote, under the new ids. Resumable and
verified-copy-up, exactly as the first pass: a runtime death loses at most
`sft_sync_every_batches` batches.

Set `RUN_GENERATION = True` to spend teacher time. It is off by default so the
notebook can be run end to end on CPU first, to read sections 3 and 4.

In [ ]:
RUN_GENERATION = False    # <- the only switch that costs GPU hours

if not RUN_GENERATION:
    print('RUN_GENERATION is False; nothing generated.')
    print(f'{len(requests):,} requests are planned and would append to the existing shards.')
elif not requests:
    print('Nothing to generate: section 3 found no headroom in any deficit category.')
else:
    from v1.quick_slm_trainer.sft import generate as G

    teacher, teacher_tok = G.load_teacher(cfg.sft)
    written = G.run_generation(
        layout=layout, cfg=cfg.sft, model=teacher, tok=teacher_tok,
        requests=requests, seed=GEN_SEED_TOPUP, progress=True,
    )
    print()
    print('new records written:', written)

## 6 · Reject additions that duplicate what was already there

This is the section the notebook exists for. Dedup within a batch is not enough:
a new example can be distinct from every other new example and still be a near-copy
of one generated last month.

`dedup_indices` keeps the **first** occurrence of any near-duplicate group. Feeding it
the existing corpus first and the additions second therefore makes existing examples
authoritative: any addition that collides with one is dropped, and only genuinely new
material survives. The fingerprint is the same one the corpus pass uses, so "duplicate"
means here exactly what it means everywhere else.

In [ ]:
from v1.quick_slm_trainer.sft.dedup import dedup_indices, example_fingerprint

# Re-read everything, now including whatever section 5 appended.
all_examples, _, _ = load_and_validate(layout, cfg.sft, progress=True)

old_ids = {id(e) for e in surviving}
additions = [e for e in all_examples if id(e) not in old_ids]

if not additions:
    print('No additions on disk yet; run section 5 first.')
else:
    ordered = list(surviving) + additions              # existing first: they win ties
    fps = [example_fingerprint(e) for e in ordered]
    keep = set(dedup_indices(fps, threshold=cfg.sft.dedup_jaccard,
                             num_perm=cfg.sft.minhash_perms, progress=True))

    base = len(surviving)
    kept_new = [ordered[i] for i in sorted(keep) if i >= base]
    dropped = len(additions) - len(kept_new)

    print()
    print(f'  additions generated      {len(additions):>8,}')
    print(f'  duplicates of existing   {dropped:>8,}  ({dropped/len(additions):.1%})')
    print(f'  genuinely new            {len(kept_new):>8,}')
    print()
    print('marginal yield by category (what the teacher time actually bought):')
    add_hist = collections.Counter(e.category for e in additions)
    new_hist = collections.Counter(e.category for e in kept_new)
    for k in sorted(add_hist):
        a, n = add_hist[k], new_hist.get(k, 0)
        print('  ' + k.ljust(24) + f'{a:>7,} generated -> {n:>7,} kept  ({n/a:.0%})')
    print()
    print('A yield near zero means that category is at its ceiling: stop generating it')
    print('and widen its seed pool or spec table instead.')

    corpus = list(surviving) + kept_new
    print()
    print('corpus after top-up:', category_histogram(corpus), ' total', len(corpus))

## 7 · Repack (off by default, writes to Drive)

Packs the combined corpus into training-ready windows, replacing what notebook 04
wrote. Same split, same masking, same packer; only the corpus differs.

Check the balance printed in section 6 before running this. If the deficit categories
barely moved, the honest next step is a code change to widen the sources rather than
another packing pass.

In [ ]:
PACK = False   # writes packed train/val windows to Drive

if not PACK:
    print('PACK is False. Set it True to write training-ready windows for notebook 05.')
elif 'corpus' not in dir():
    print('Nothing to pack: run section 6 first.')
else:
    from v1.quick_slm_trainer.sft.pack import pack_split, split_examples, write_stats
    from v1.quick_slm_trainer.sft.corpus import paired_integrity, subtype_histogram
    from v1.quick_slm_trainer.tokenizer import load_tokenizer

    tok = load_tokenizer(layout.tokenizer_dir, patch=False)
    cfg.sft.ctx = cfg.data.ctx

    train_ex, val_ex = split_examples(corpus, val_fraction=cfg.sft.val_fraction)
    for name, split in (('train', train_ex), ('val', val_ex)):
        _, bad = paired_integrity(split)
        assert not bad, name + ' split has a lone branch: ' + str(bad[:5])
    print('train', len(train_ex), '  val', len(val_ex))

    train_stats = pack_split(layout, 'train', train_ex, tok, cfg.sft)
    print()
    val_stats = pack_split(layout, 'val', val_ex, tok, cfg.sft)

    path = write_stats(layout, {
        'source': 'v1 corpus plus the 04c top-up pass',
        'config': cfg.sft.to_dict(),
        'topup': {'tag': TOPUP_TAG, 'plan_seed': PLAN_SEED_TOPUP, 'gen_seed': GEN_SEED_TOPUP},
        'after_topup': {'examples': len(corpus), 'by_category': category_histogram(corpus),
                        'by_subtype': subtype_histogram(corpus)},
        'pack': {'train': train_stats.to_dict(), 'val': val_stats.to_dict()},
    })
    print()
    print('wrote', path)
    print('packed train tokens:', f'{train_stats.total_tokens:,}')
    print('packed val tokens  :', f'{val_stats.total_tokens:,}')